# HDB resale — HistGradientBoosting V2

**Target:** `log1p(resale_price)` via `TransformedTargetRegressor` — metrics and predictions are in **dollar scale** (`expm1`).

**Tuning:** `RandomizedSearchCV` with **`TimeSeriesSplit`** on data sorted by `Tranc_YearMonth` (walk-forward style folds).

Engineered features: **`postal_sector`**, **`storey_ratio`**, **`rental_ratio`**, **`rooms_num`**, **`month_index`** (`(Tranc_Year - 2000) * 12 + Tranc_Month`).

**Submissions:**
- `ROOT / submission / sub_hgb_v2_t.csv` (time split)
- `ROOT / submission / sub_hgb_v2_r.csv` (random split)


In [1]:
# Paths — submissions: ROOT / submission / sub_hgb_v2_{t,r}.csv
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebook" else _cwd
TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUB_PATH = ROOT / "data" / "sample_sub_reg.csv"
SUBMISSION_PATH_T = ROOT / "submission" / "sub_hgb_v2_t.csv"
SUBMISSION_PATH_R = ROOT / "submission" / "sub_hgb_v2_r.csv"

print(f"ROOT: {ROOT.resolve()}")
print(f"Train: {TRAIN_PATH.resolve()}")
print(f"Test:  {TEST_PATH.resolve()}")
print(f"Time-split submission:   {SUBMISSION_PATH_T.resolve()}")
print(f"Random-split submission: {SUBMISSION_PATH_R.resolve()}")

train = pd.read_csv(TRAIN_PATH, low_memory=False)
test = pd.read_csv(TEST_PATH, low_memory=False)
print(train.shape, test.shape)


ROOT: /Users/ian/Documents/NTU/DSAI/Module 3/HDB Kaggle
Train: /Users/ian/Documents/NTU/DSAI/Module 3/HDB Kaggle/data/train.csv
Test:  /Users/ian/Documents/NTU/DSAI/Module 3/HDB Kaggle/data/test.csv
Submission will be written to: /Users/ian/Documents/NTU/DSAI/Module 3/HDB Kaggle/submission/sub_hgb_v2.csv
(150634, 77) (16737, 76)


In [2]:
import numpy as np
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

RNG = 42
TARGET = "resale_price"



In [3]:
ROOMS_FROM_FLAT = {
    "1 ROOM": 1,
    "2 ROOM": 2,
    "3 ROOM": 3,
    "4 ROOM": 4,
    "5 ROOM": 5,
    "EXECUTIVE": 6,
    "MULTI-GENERATION": 7,
}


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    pc = out["postal"].astype(str).str.replace(r"\.0$", "", regex=True)
    out["postal_sector"] = pd.to_numeric(pc.str.slice(0, 2), errors="coerce")

    ms = pd.to_numeric(out["mid_storey"], errors="coerce")
    mx = pd.to_numeric(out["max_floor_lvl"], errors="coerce")
    out["storey_ratio"] = np.where(mx > 0, ms / mx, np.nan)

    rcols = ["1room_rental", "2room_rental", "3room_rental", "other_room_rental"]
    total_rent = np.zeros(len(out))
    for c in rcols:
        total_rent += pd.to_numeric(out[c], errors="coerce").fillna(0).to_numpy(dtype=float)
    td = pd.to_numeric(out["total_dwelling_units"], errors="coerce").to_numpy(dtype=float)
    out["rental_ratio"] = np.where(td > 0, total_rent / td, np.nan)

    out["rooms_num"] = out["flat_type"].map(ROOMS_FROM_FLAT).astype(float)
    ty = pd.to_numeric(out["Tranc_Year"], errors="coerce")
    tm = pd.to_numeric(out["Tranc_Month"], errors="coerce")
    out["month_index"] = (ty - 2000) * 12 + tm
    return out


train = add_engineered_features(train)
test = add_engineered_features(test)

DROP_FEATURES = [
    "id",
    "floor_area_sqft",
    "postal",
    "address",
    "block",
    "street_name",
    "flat_type",
    "flat_model",
    "1room_sold",
    "2room_sold",
    "3room_sold",
    "4room_sold",
    "5room_sold",
    "exec_sold",
    "multigen_sold",
    "studio_apartment_sold",
    "1room_rental",
    "2room_rental",
    "3room_rental",
    "other_room_rental",
    "bus_stop_name",
    "sec_sch_name",
]

feature_cols = [c for c in train.columns if c not in DROP_FEATURES and c != TARGET]
missing_in_test = set(feature_cols) - set(test.columns)
missing_in_train = set(test.columns) - set(train.columns) - {TARGET}
assert not missing_in_test, f"Features missing in test: {missing_in_test}"
print("Columns only in test (expected: none except target):", missing_in_train)
print(
    f"Features: {len(feature_cols)} columns (incl. postal_sector, storey_ratio, rental_ratio, rooms_num, month_index)"
)

X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y = train[TARGET].astype(float)

period = pd.to_datetime(train["Tranc_YearMonth"], format="%Y-%m")



Columns only in test (expected: none except target): set()
Features: 58 columns (incl. postal_sector, storey_ratio, rental_ratio, rooms_num)


In [4]:
# Preprocess + HGBR (inner pipeline); outer wrapper applies log1p / expm1 on target
num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]
cat_low = [c for c in cat_cols if X_train[c].nunique(dropna=True) <= 255]
cat_high = [c for c in cat_cols if c not in cat_low]
print(
    f"Numeric: {len(num_cols)}, Categorical (<=255 levels): {len(cat_low)}, "
    f"Ordinal-as-numeric: {len(cat_high)}"
)
if cat_high:
    print("  High-cardinality object columns:", cat_high)

num_pipe = SimpleImputer(strategy="median")
cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        ),
    ]
)

transformers = [("num", num_pipe, num_cols)]
if cat_low:
    transformers.append(("cat_low", cat_pipe, cat_low))
if cat_high:
    transformers.append(("cat_high", cat_pipe, cat_high))

preprocess = ColumnTransformer(
    transformers=transformers,
    verbose_feature_names_out=False,
)

cat_idx = list(range(len(num_cols), len(num_cols) + len(cat_low)))
hgb_base_kw = dict(random_state=RNG, max_iter=350, learning_rate=0.08, max_depth=12)
if cat_idx:
    hgb_base_kw["categorical_features"] = cat_idx

inner_pipe = Pipeline(
    steps=[
        ("prep", preprocess),
        ("hgb", HistGradientBoostingRegressor(**hgb_base_kw)),
    ]
)

model = TransformedTargetRegressor(
    regressor=inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1,
)

# Hyperparameter search (names refer to inner Pipeline: regressor__hgb__*)
param_distributions = {
    "regressor__hgb__learning_rate": [0.05, 0.07, 0.09, 0.11, 0.13],
    "regressor__hgb__max_depth": [8, 10, 12, 14],
    "regressor__hgb__max_iter": [250, 300, 350, 400, 450],
    "regressor__hgb__min_samples_leaf": [15, 20, 25, 30],
    "regressor__hgb__l2_regularization": [0.01, 0.05, 0.1, 0.2, 0.3],
}

tscv = TimeSeriesSplit(n_splits=4)
order = np.argsort(period.values)
X_ord = X_train.iloc[order].reset_index(drop=True)
y_ord = y.iloc[order].reset_index(drop=True)

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=18,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    random_state=RNG,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

search.fit(X_ord, y_ord)



Numeric: 46, Categorical (<=255 levels): 12, Ordinal-as-numeric: 0
Fitting 4 folds for each of 18 candidates, totalling 72 fits


Exception ignored in: <function ResourceTracker.__del__ at 0x106e72520>
Traceback (most recent call last):
  File "/Users/ian/miniconda3/envs/hdb-ml-env/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/Users/ian/miniconda3/envs/hdb-ml-env/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/Users/ian/miniconda3/envs/hdb-ml-env/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x109172520>
Traceback (most recent call last):
  File "/Users/ian/miniconda3/envs/hdb-ml-env/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/Users/ian/miniconda3/envs/hdb-ml-env/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/Users/ian/miniconda3/envs/hdb-ml-env/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessErr

RandomizedSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=4, test_size=None),
                   estimator=TransformedTargetRegressor(func=<ufunc 'log1p'>,
                                                        inverse_func=<ufunc 'expm1'>,
                                                        regressor=Pipeline(steps=[('prep',
                                                                                   ColumnTransformer(transformers=[('num',
                                                                                                                    SimpleImputer(strategy='median'),
                                                                                                                    ['floor_area_sqm',
                                                                                                                     'lease_commence_date',
                                                                                                                     'Tranc_Year',
                                                                                                                     'Tranc_Month',
                                                                                                                     '...
                   n_iter=18, n_jobs=-1,
                   param_distributions={'regressor__hgb__l2_regularization': [0.01,
                                                                              0.05,
                                                                              0.1,
                                                                              0.2,
                                                                              0.3],
                                        'regressor__hgb__learning_rate': [0.05,
                                                                          0.07,
                                                                          0.09,
                                                                          0.11,
                                                                          0.13],
                                        'regressor__hgb__max_depth': [8, 10, 12,
                                                                      14],
                                        'regressor__hgb__max_iter': [250, 300,
                                                                     350, 400,
                                                                     450],
                                        'regressor__hgb__min_samples_leaf': [15,
                                                                             20,
                                                                             25,
                                                                             30]},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=1)

In [5]:
# Report tuning results (RMSE is in dollar space thanks to TransformedTargetRegressor)
print("Best params:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")
best_cv_rmse = -search.best_score_
print(f"Best CV RMSE (mean across time splits, dollar scale): {best_cv_rmse:,.2f}")
cutoff = period.max() - pd.DateOffset(months=12)
val_mask = period >= cutoff
y_val_pred = search.best_estimator_.predict(X_train.loc[val_mask])
rmse_hold = root_mean_squared_error(y[val_mask], y_val_pred)
mae_hold = mean_absolute_error(y[val_mask], y_val_pred)
print(f"Last-12-months holdout RMSE (primary): {rmse_hold:,.2f}")
print(f"Last-12-months holdout MAE: {mae_hold:,.2f}")
from sklearn.base import clone
X_tr_r, X_va_r, y_tr_r, y_va_r = train_test_split(
    X_train,
    y,
    test_size=0.30,
    random_state=RNG,
    shuffle=True,
)
model_rand_eval = clone(search.best_estimator_)
model_rand_eval.fit(X_tr_r, y_tr_r)
y_va_r_pred = model_rand_eval.predict(X_va_r)
rmse_r = root_mean_squared_error(y_va_r, y_va_r_pred)
mae_r = mean_absolute_error(y_va_r, y_va_r_pred)
print(f"Random 70/30 holdout RMSE: {rmse_r:,.2f}")
print(f"Random 70/30 holdout MAE: {mae_r:,.2f}")


Best params:
  regressor__hgb__min_samples_leaf: 15
  regressor__hgb__max_iter: 450
  regressor__hgb__max_depth: 10
  regressor__hgb__learning_rate: 0.09
  regressor__hgb__l2_regularization: 0.2
Best CV RMSE (mean across time splits, dollar scale): 44,428.40
Last-12-months holdout RMSE (primary): 21,683.22
Last-12-months holdout MAE: 15,738.87


In [6]:
# Fit split-specific submissions using best tuned estimator
from sklearn.base import clone

# Time split model (same holdout boundary as metrics)
final_model_t = clone(search.best_estimator_)
final_model_t.fit(X_train.loc[~val_mask], y.loc[~val_mask])
test_pred_t = final_model_t.predict(X_test)

# Random split model (70/30 train fold)
X_tr_r, _, y_tr_r, _ = train_test_split(
    X_train,
    y,
    test_size=0.30,
    random_state=RNG,
    shuffle=True,
)
final_model_r = clone(search.best_estimator_)
final_model_r.fit(X_tr_r, y_tr_r)
test_pred_r = final_model_r.predict(X_test)

sample = pd.read_csv(SAMPLE_SUB_PATH, nrows=5)
sub_t = pd.DataFrame({"Id": test["id"], "Predicted": test_pred_t})
sub_r = pd.DataFrame({"Id": test["id"], "Predicted": test_pred_r})
assert list(sub_t.columns) == list(sample.columns), (sub_t.columns.tolist(), sample.columns.tolist())
assert list(sub_r.columns) == list(sample.columns), (sub_r.columns.tolist(), sample.columns.tolist())

SUBMISSION_PATH_T.parent.mkdir(parents=True, exist_ok=True)
sub_t.to_csv(SUBMISSION_PATH_T, index=False)
sub_r.to_csv(SUBMISSION_PATH_R, index=False)
print(f"Wrote {SUBMISSION_PATH_T.resolve()} ({len(sub_t):,} rows)")
print(f"Wrote {SUBMISSION_PATH_R.resolve()} ({len(sub_r):,} rows)")
print(sub_t.head())


Wrote /Users/ian/Documents/NTU/DSAI/Module 3/HDB Kaggle/submission/sub_hgb_v2.csv (16,737 rows)
       Id      Predicted
0  114982  371811.324572
1   95653  460727.004157
2   40303  357937.424386
3  109506  288241.271364
4  100149  417566.666832


Exception ignored in: <function ResourceTracker.__del__ at 0x103572520>
Traceback (most recent call last):
  File "/Users/ian/miniconda3/envs/hdb-ml-env/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/Users/ian/miniconda3/envs/hdb-ml-env/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/Users/ian/miniconda3/envs/hdb-ml-env/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x108e72520>
Traceback (most recent call last):
  File "/Users/ian/miniconda3/envs/hdb-ml-env/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/Users/ian/miniconda3/envs/hdb-ml-env/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/Users/ian/miniconda3/envs/hdb-ml-env/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessErr

## Next steps

- Map rare **`flat_type`** values (e.g. `STUDIO APARTMENT`) in `ROOMS_FROM_FLAT`.
- Expand search space or use **`BayesSearchCV`** for cheaper optimization.
- Blend with **LightGBM** / **XGBoost**; **SHAP** on a single time-based validation fold.

